In [1]:
import glob
import pandas as pd
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.testing.print_coercion_tables import print_new_cast_table
from scipy.stats import gmean
import scipy.stats as stats
import re
import os

from functools import reduce
from matplotlib.colors import to_rgb, to_hex
import colorsys

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

In [2]:
fname= "./dynamic_results_eager.txt"
dfdyn = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="skip")
#df = df.drop(columns=['best', 'minuses'])
print(dfdyn)


                            vertex    start_time   finish_time  proc_num
0            get_software_versions      0.000000     20.978000         5
1             output_documentation      0.000000     20.978000         1
2                           kraken     20.978000     41.956000         5
3      lanemerge_hostremoval_fastq     20.978000     41.956000         1
4                    indexinputbam      0.000000     55.941333         3
5                       convertBam     41.956000     62.934000         5
6                             malt     41.956000     62.934000         1
7                  unzip_reference     62.934000     83.912000         5
8                     kraken_parse     83.912000    104.890000         5
9                            fastp    104.890000    125.868000         5
10                          fastqc    125.868000   1980.466250         5
11                     maltextract     62.934000     83.912000         1
12                    makeBWAIndex   1980.466250   

In [31]:
fname= "./static_results_eager.txt"
dfstat = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="skip")
#df = df.drop(columns=['best', 'minuses'])
print(dfstat)

                            vertex    start_time   finish_time  proc_num
0                       convertBam      0.000000     20.978000         5
1                            fastp     20.978000     41.956000         5
2                  adapter_removal     41.956000   5957.601750         5
3                  unzip_reference      0.000000     20.978000         1
4                     makeBWAIndex     20.978000     41.956000         1
5                        lanemerge   5957.601750   5978.579750         5
6                              bwa   5978.579750  29272.637500         5
7                circulargenerator     41.956000     62.934000         1
8                    indexinputbam      0.000000     55.941333         3
9                   circularmapper  29272.637500  29293.615500         5
10                          bwamem  29293.615500  29314.593500         5
11                         bowtie2  29314.593500  29335.571500         5
12                   seqtype_merge  29335.571500  2

In [3]:
def byProcessor(somedf):
    for proc, df_proc in somedf.groupby("proc_num"):
        print(f"\n=== Processor {proc} ===")
        df_sorted = df_proc.sort_values("start_time")

        for _, r in df_sorted.iterrows():
            print(f"{r['start_time']:>10.3f}  {r['finish_time']:>10.3f}  {r['vertex']}")


#for proc, df_proc in dfdyn.groupby("proc_num"):
#    print(f"\n=== Processor {proc} ===")
#    df_sorted = df_proc.sort_values("start_time")#

#    for _, r in df_sorted.iterrows():
#        print(f"{r['start_time']:>10.3f}  {r['finish_time']:>10.3f}  {r['vertex']}")

byProcessor(dfdyn)


=== Processor 0 ===
     0.000      37.156  get_software_versions_00001786
    37.156      74.312  output_documentation_00001699
    74.312     111.468  get_software_versions_00001606
   111.468     148.623  get_software_versions_00001570
   148.623     185.779  get_software_versions_00001534
   185.779     222.935  get_software_versions_00001498
   222.935     260.091  get_software_versions_00001462
   260.091   15786.542  fastqc_00001436
 15786.542   15823.698  get_software_versions_00001078
 15823.698   15860.854  output_documentation_00001063
 15860.854   15898.010  output_documentation_00001039
 15898.010   15935.166  get_software_versions_00001006
 15935.166   15972.322  output_documentation_00000979
 15972.322   16009.478  get_software_versions_00000946
 16009.478   16046.633  get_software_versions_00000910
 16046.633   16083.789  get_software_versions_00000874
 16083.789   16120.945  get_software_versions_00000838
 16120.945   16158.101  get_software_versions_00000802
 16158.1

In [33]:

byProcessor(dfstat)


=== Processor 1 ===
     0.000      20.978  unzip_reference
    20.978      41.956  makeBWAIndex
    41.956      62.934  circulargenerator
    62.934      83.912  makeSeqDict
    83.912     104.890  makeFastaIndex
   104.890     125.868  lanemerge_hostremoval_fastq
   125.868     146.846  mask_reference_for_pmdtools

=== Processor 2 ===
     0.000     111.883  malt
   111.883     223.765  maltextract

=== Processor 3 ===
     0.000      55.941  indexinputbam
    55.941     111.883  get_software_versions
   111.883     167.824  output_documentation

=== Processor 4 ===
     0.000      83.912  kraken
    83.912     167.824  kraken_parse
   167.824     251.736  kraken_merge

=== Processor 5 ===
     0.000      20.978  convertBam
    20.978      41.956  fastp
    41.956    5957.602  adapter_removal
  5957.602    5978.580  lanemerge
  5978.580   29272.638  bwa
 29272.638   29293.615  circularmapper
 29293.615   29314.593  bwamem
 29314.593   29335.571  bowtie2
 29335.571   29356.550  seqty

In [27]:
# compute durations
dfdyn["duration_dyn"] = dfdyn["finish_time"] - dfdyn["start_time"]
dfstat["duration_stat"] = dfstat["finish_time"] - dfstat["start_time"]

# rename processors
dfdyn = dfdyn.rename(columns={"proc_num": "processor_dyn"})
dfstat = dfstat.rename(columns={"proc_num": "processor_stat"})

# merge
merged = dfdyn.merge(dfstat, on="vertex", suffixes=("_dyn", "_stat"))

# find mismatches
diff = merged[ (merged["duration_dyn"] != merged["duration_stat"] )]# & (merged["processor_dyn"] == merged["processor_stat"])  ]

# pick columns to show
diff = diff[[
    "vertex",
    "duration_dyn",
    "processor_dyn",
    "duration_stat",
    "processor_stat"
]]

print(diff)

                            vertex  duration_dyn  processor_dyn  \
0            get_software_versions     20.978000              5   
1             output_documentation     20.978000              1   
2                           kraken     20.978000              5   
3      lanemerge_hostremoval_fastq     20.978000              1   
5                       convertBam     20.978000              5   
6                             malt     20.978000              1   
7                  unzip_reference     20.978000              5   
8                     kraken_parse     20.978000              5   
9                            fastp     20.978000              5   
10                          fastqc   1854.598250              5   
11                     maltextract     20.978000              1   
12                    makeBWAIndex     20.978000              5   
13               circulargenerator     20.978000              5   
14                  makeFastaIndex     20.978000              

In [37]:
fname= "./dynamic_results_atacseq.txt"
dfdyn = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="skip")
#df = df.drop(columns=['best', 'minuses'])
#print(dfdyn)
fname= "./static_results_atacseq.txt"
dfstat = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="skip")
#df = df.drop(columns=['best', 'minuses'])
#print(dfstat)

byProcessor(dfdyn)
print("--------------------------")
byProcessor(dfstat)


=== Processor 30 ===
     0.000   58022.363  CHECK_DESIGN
 58022.363  668136.425  TRIMGALORE
668136.425  908829.550  FASTQC
908829.550  2220030.487  BWA_MEM
2220030.487  2494927.263  SORT_BAM
2494927.263  3416329.587  MERGED_LIB_BAM
3416329.587  4095709.612  MERGED_LIB_BAM_FILTER
4095709.612  4487844.100  MERGED_LIB_BAM_REMOVE_ORPHAN
4487844.100  4872804.287  MERGED_LIB_MACS2
4872855.265  5073354.227  MERGED_LIB_BIGWIG
5073354.227  5275942.327  MERGED_LIB_PICARD_METRICS
5275942.327  5333964.690  MERGED_LIB_PLOTFINGERPRINT
5334020.435  5498967.160  MERGED_LIB_ATAQV
5498967.160  5562937.122  MERGED_LIB_MACS2_ANNOTATE
5562937.122  5620959.485  MERGED_LIB_CONSENSUS_COUNTS
5620959.485  5678981.847  MERGED_LIB_ATAQV_MKARV
5678981.847  5737004.210  MERGED_LIB_MACS2_QC
5737004.210  5795026.572  MERGED_LIB_CONSENSUS_DESEQ2
5795028.910  5853051.273  MULTIQC
5853064.702  5911087.064  IGV

=== Processor 31 ===
     0.000   58022.363  BWA_INDEX

=== Processor 32 ===
     0.000   58022.363  MAKE_GE

In [4]:
fname= "./dynamic_results_methylseq_2000.txt"
dfdyn = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="skip")
#df = df.drop(columns=['best', 'minuses'])
#print(dfdyn)
fname= "./static_results_methylseq_2000.txt"
dfstat = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="skip")
#df = df.drop(columns=['best', 'minuses'])
#print(dfstat)

byProcessor(dfdyn)
print("--------------------------")
byProcessor(dfstat)


=== Processor 0 ===
     0.000      37.156  get_software_versions_00001786
    37.156      74.312  output_documentation_00001699
    74.312     111.468  get_software_versions_00001606
   111.468     148.623  get_software_versions_00001570
   148.623     185.779  get_software_versions_00001534
   185.779     222.935  get_software_versions_00001498
   222.935     260.091  get_software_versions_00001462
   260.091   15786.542  fastqc_00001436
 15786.542   15823.698  get_software_versions_00001078
 15823.698   15860.854  output_documentation_00001063
 15860.854   15898.010  output_documentation_00001039
 15898.010   15935.166  get_software_versions_00001006
 15935.166   15972.322  output_documentation_00000979
 15972.322   16009.478  get_software_versions_00000946
 16009.478   16046.633  get_software_versions_00000910
 16046.633   16083.789  get_software_versions_00000874
 16083.789   16120.945  get_software_versions_00000838
 16120.945   16158.101  get_software_versions_00000802
 16158.1

In [3]:
%run ChartsCommon.ipynb
labels = ['A1', 'A2', 'A3', 'BASE']

path = "./ruben-werte/final-kinda/merged/*.txt"
print(path)

patterndevs = r'^(BASE|A\d+)-(ndev)'

dfs=read_dfs(path,patterndevs, 2)

dfsVar1 = [dfs[('A1','ndev')], dfs[('A2','ndev')], dfs[('A3','ndev')], dfs[('BASE','ndev')]]
#print(dfsVar1)
merged_df_var1 = merge_correct_columns(dfsVar1, labels)
plot_df_old = buld_plot_df(merged_df_var1)
plot_df_old = plot_df_old[plot_df_old["size"]!= 18000]

print("----------------------------------------------")
print(plot_df_old.info)

print("----------------------------------------------")

path = "./results-3-12/merged/*.txt"
print(path)

patterndevs = r'^(BASE|A\d+)-(ndev)'

dfs=read_dfs(path,patterndevs, 2)

dfsVar1 = [dfs[('A1','ndev')], dfs[('A2','ndev')], dfs[('A3','ndev')], dfs[('BASE','ndev')]]
#print(dfsVar1)
merged_df_var1 = merge_correct_columns(dfsVar1, labels)
plot_df_new = buld_plot_df(merged_df_var1)
plot_df_new = plot_df_new[plot_df_new["size"]!= 18000]

print(plot_df)

./ruben-werte/final-kinda/merged/*.txt
----------------------------------------------
<bound method DataFrame.info of        size              wf_name     inp_size      relation     ratio  \
0     30000    atacseq_30000.dot  14091675276      internal  1.301302   
1     15000    chipseq_15000.dot  41366257414      internal  1.307672   
2     15000  methylseq_15000.dot   6761426956      internal  0.898391   
3       100                eager  19075314980      internal  0.545767   
4      1000       eager_1000.dot  25705994498      internal  1.057662   
...     ...                  ...          ...           ...       ...   
2272    100              atacseq   2223941232  stat_vs_base  0.805369   
2273    100              atacseq   3908761308  stat_vs_base  0.824926   
2274    100              atacseq  11187378708  stat_vs_base  0.933790   
2275    100              atacseq  11809629756  stat_vs_base  0.929529   
2276    100              atacseq  14091675276  stat_vs_base  1.042867   

     

NameError: name 'plot_df' is not defined

In [34]:
plot_df_old2 = plot_df_old.rename(columns={"ratio": "ratio_old"})
plot_df_new2 = plot_df_new.rename(columns={"ratio": "ratio_new"})
merged_plots_dfs = plot_df_old2.merge(
    plot_df_new2,
    on=["wf_name", "inp_size", "algorithm", "relation"],
   # how="outer"
    how="inner"
)

diff = merged_plots_dfs[merged_plots_dfs["ratio_old"] != merged_plots_dfs["ratio_new"]]
#print(diff[["wf_name","size_x", "relation", "ratio_old", "ratio_new"]])
#diff[["wf_name","size_x", "relation", "ratio_old", "ratio_new"]].to_csv("not_equal_ratios.csv", index=False)

diff_correct_cols= diff[["wf_name","inp_size", "size_x", "relation", "ratio_old", "ratio_new"]]

#print(diff_correct_cols[diff_correct_cols["ratio_old"]<diff_correct_cols["ratio_new"]])
print(diff_correct_cols[diff_correct_cols["relation"] == "stat_vs_base"])
stat_vs_base_rels= diff_correct_cols[diff_correct_cols["relation"] == "stat_vs_base"]
print(stat_vs_base_rels[stat_vs_base_rels["size_x"]<1000])


#same = merged_plots_dfs[merged_plots_dfs["ratio_old"] == merged_plots_dfs["ratio_new"]]
#print("-----------------------------------\n Same: \n", same)
#same.to_csv("equal_ratios.csv", index=False)

                  wf_name     inp_size  size_x      relation    ratio_old  \
482     atacseq_30000.dot  14091675276   30000  stat_vs_base  5159.513264   
483     chipseq_15000.dot  41366257414   15000  stat_vs_base  1592.410410   
484   methylseq_15000.dot   6761426956   15000  stat_vs_base   120.453235   
485        eager_1000.dot  25705994498    1000  stat_vs_base  2079.474207   
486       eager_20000.dot   8330435694   20000  stat_vs_base  9258.240122   
...                   ...          ...     ...           ...          ...   
2163  methylseq_25000.dot  22500260922   25000  stat_vs_base   812.601269   
2164    chipseq_30000.dot   3793245764   30000  stat_vs_base  6352.195862   
2165   methylseq_1000.dot   8885797810    1000  stat_vs_base    99.796692   
2166  methylseq_10000.dot  17027257018   10000  stat_vs_base   159.696540   
2168    chipseq_10000.dot   4807293396   10000  stat_vs_base  8702.253677   

      ratio_new  
482    5.202210  
483    1.180069  
484    0.657512  
485

In [4]:
def byProcessor(somedf, proc_column_name, start_time_column_name, finish_time_column_name):
    for proc, df_proc in somedf.groupby(proc_column_name):
        print(f"\n=== Processor {proc} ===")
        df_sorted = df_proc.sort_values(start_time_column_name)

        for _, r in df_sorted.iterrows():
            print(f"{r[start_time_column_name]:>10.3f}  {r[finish_time_column_name]:>10.3f}  {r['vertex']}")
            
fname= "./bl_vs_heft.txt"
bl_and_heft = pd.read_csv(fname, delimiter='\t', header=0, on_bad_lines="warn")
print(bl_and_heft.head())

byProcessor(bl_and_heft, "proc_bl", "start_bl", "finish_bl")

print("------------------------------")

byProcessor(bl_and_heft, "proc_heft", "start_heft", "finish_heft")

                 vertex  start_bl   finish_bl  proc_bl  variant_bl  \
0  trim_galore_00000192       0.0  8887618.95       35           1   
1  trim_galore_00000168       0.0  8887618.95       34           1   
2  trim_galore_00000072       0.0  8887618.95       33           1   
3  trim_galore_00000084       0.0  8887618.95       32           1   
4  trim_galore_00000096       0.0  8887618.95       31           1   

                vertex1  start_heft  finish_heft  proc_heft  variant_heft  
0  trim_galore_00000192         0.0   8887618.95         35             1  
1  trim_galore_00000168         0.0   8887618.95         34             1  
2  trim_galore_00000072         0.0   8887618.95         33             1  
3  trim_galore_00000084         0.0   8887618.95         32             1  
4  trim_galore_00000096         0.0   8887618.95         31             1  

=== Processor 0 ===
     0.000    1462.400  output_documentation_00000187
  1462.400    2924.800  output_documentation_000

In [5]:
#group = proc // 6
bl_and_heft["group_bl"] = bl_and_heft["proc_bl"] // 6
bl_and_heft["group_heft"] = bl_and_heft["proc_heft"] // 6
diff_groups = bl_and_heft[bl_and_heft["group_bl"] != bl_and_heft["group_heft"]]
print(diff_groups[[
    "vertex", "proc_bl", "group_bl", 
    "proc_heft", "group_heft"
]])

                            vertex  proc_bl  group_bl  proc_heft  group_heft
48    bismark_deduplicate_00000189       11         1         35           5
50    bismark_deduplicate_00000045       11         1         34           5
56    bismark_deduplicate_00000141       33         5         10           1
58    bismark_deduplicate_00000033        6         1         32           5
59    bismark_deduplicate_00000006       32         5         10           1
..                             ...      ...       ...        ...         ...
183               multiqc_00000041       11         1         34           5
184  output_documentation_00000163        0         0         16           2
185               multiqc_00000185       10         1         35           5
187  output_documentation_00000031        5         0         17           2
190               multiqc_00000029        8         1         32           5

[87 rows x 5 columns]


In [6]:
def compare_by_processor_full(df):
    processors = sorted(set(df["proc_bl"]) | set(df["proc_heft"]))

    for proc in processors:
        print(f"\n=== Processor {proc} ===")

        # Extract BL rows on this processor
        bl = df[df["proc_bl"] == proc][[
            "vertex", "start_bl", "finish_bl"
        ]].rename(columns={
            "start_bl": "start",
            "finish_bl": "finish",
        })
        bl["source"] = "BL"

        # Extract HEFT rows on this processor
        hf = df[df["proc_heft"] == proc][[
            "vertex1", "start_heft", "finish_heft"
        ]].rename(columns={
            "vertex1": "vertex",
            "start_heft": "start",
            "finish_heft": "finish",
        })
        hf["source"] = "HEFT"

        # Combine BL + HEFT
        merged = (
            pd.merge(
                bl, hf,
                on=["vertex", "start", "finish"],
                how="outer",
                suffixes=("_bl", "_heft")
            )
        )

        # Sort by start time
        merged = merged.sort_values("start")

        # Pretty print
        for _, r in merged.iterrows():
            print(f"{r.get('start', float('nan')):>10.3f}  "
                  f"{r.get('finish', float('nan')):>10.3f}  "
                  f"{r['vertex']:<40}  "
                  f"{'BL' if not pd.isna(r.get('source_bl')) else ''} "
                  f"{'HEFT' if not pd.isna(r.get('source_heft')) else ''}")
            
compare_by_processor_full(bl_and_heft)


=== Processor 0 ===
     0.000    1462.400  output_documentation_00000187             BL 
     0.000    1462.400  output_documentation_00000139              HEFT
  1462.400    2924.800  output_documentation_00000055             BL 
  2924.800    4387.200  output_documentation_00000163             BL 

=== Processor 1 ===
     0.000    1462.400  get_software_versions_00000070            BL 
     0.000    1462.400  output_documentation_00000187              HEFT
  1462.400    2924.800  output_documentation_00000091             BL 
  2924.800    4387.200  output_documentation_00000019             BL 

=== Processor 2 ===
     0.000    1462.400  get_software_versions_00000142            BL 
     0.000    1462.400  get_software_versions_00000070             HEFT
  1462.400    2924.800  output_documentation_00000151             BL 
  2924.800    4387.200  output_documentation_00000043             BL 

=== Processor 3 ===
     0.000    1462.400  get_software_versions_00000082            BL 


In [36]:
fname= "./methylseq_1000_dynamic_starts_and_finishes.txt"
dyn_methylseq = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="warn")
#print(dyn_methylseq.head())

# Pivot start/finish times
df_wide = (
    dyn_methylseq.pivot(index="vertex", columns="startOrFinish", values="time")
      .rename(columns={"start": "start_time", "finish": "end_time"})
      .reset_index()
)

# Add processor where start happens
df_wide = df_wide.merge(
    dyn_methylseq[dyn_methylseq["startOrFinish"] == "start"][["vertex", "proc"]], 
    on="vertex"
)


df_wide["duration"] = df_wide["end_time"] - df_wide["start_time"]
df_wide2 = df_wide.rename(columns={"proc": "processor",
                                 "duration": "duration"})

#print(df_wide.head())

fname= "./methylseq_1000_static.txt"
stat_methylseq = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="warn")
stat_methylseq["duration"] = stat_methylseq["finish"] - stat_methylseq["start"]

stat_methylseq2 = stat_methylseq.rename(columns={"start": "start_time",
                                 "finish": "end_time"})

#print(stat_methylseq.head())


merged = df_wide2.merge(
    stat_methylseq2,
    on="vertex",
    suffixes=("_dyn", "_stat")
)
#print(merged.head())


# Rows where duration differs
diff = merged[ merged["duration_dyn"] != merged["duration_stat"] ]

# Sort by dynamic start time
diff = diff.sort_values(by="start_time_dyn")

res = diff[[
    "vertex",
    "start_time_dyn",
    "duration_dyn", "processor_dyn",
    "start_time_stat",
    "duration_stat", "processor_stat"
]]

#print(res)


#res.to_csv("dyn_and_stat_methylseq_compared.csv", index=False)

same_proc_diff_duration = merged[
    (merged["duration_dyn"] != merged["duration_stat"]) &
    (merged["processor_dyn"] == merged["processor_stat"])
]

# Sort by start time (dynamic start)
same_proc_diff_duration = same_proc_diff_duration.sort_values(by="start_time_dyn")

print(same_proc_diff_duration[[
    "vertex",
    "processor_dyn",
    "duration_dyn",
    "duration_stat",
    "start_time_dyn",
    "start_time_stat"
]])

same_proc_diff_duration[[
    "vertex",
    "processor_dyn",
    "duration_dyn",
    "duration_stat",
    "start_time_dyn",
    "start_time_stat"
]].to_csv("same_proc_diff_duration.csv", index=False)


                             vertex  processor_dyn  duration_dyn  \
990            trim_galore_00000984             31  8.887619e+06   
566  get_software_versions_00000862             12  9.749333e+02   
557  get_software_versions_00000754              1  1.462400e+03   
989            trim_galore_00000972             34  8.887619e+06   
480                 fastqc_00000824             19  2.916333e+06   
553  get_software_versions_00000706             13  9.749333e+02   
542  get_software_versions_00000574              0  1.462400e+03   
700   output_documentation_00000475             13  9.749333e+02   
691   output_documentation_00000367             16  9.749333e+02   
515  get_software_versions_00000250              0  1.462400e+03   
682   output_documentation_00000259              1  1.462400e+03   
668   output_documentation_00000091              3  1.462400e+03   
498  get_software_versions_00000046              2  1.462400e+03   
986            trim_galore_00000936             

In [31]:
def print_by_processor(df, label):
    print(f"\n===== {label.upper()} =====\n")
    for proc in sorted(df["processor"].unique()):
        print(f"\n--- Processor {proc} ---")
        block = df[df["processor"] == proc].sort_values(by="start_time")

        for _, row in block.iterrows():
            print(
                f"{row['start_time']:12.3f} → {row['end_time']:12.3f}   "
                f"{row['vertex']}   (duration {row['duration']:.3f})"
            )


In [32]:
print_by_processor(stat_methylseq2, "STATIC")
print_by_processor(df_wide2, "DYNAMIC")


===== STATIC =====


--- Processor 0 ---
       0.000 →  8748999.500   fastqc_00000308   (duration 8748999.500)
624945664.788 → 625006784.188   bismark_methXtract_00000640   (duration 61119.400)
625006784.188 → 625026694.788   qualimap_00000841   (duration 19910.600)
625026694.788 → 625046605.388   qualimap_00000865   (duration 19910.600)
625046605.388 → 625066515.988   qualimap_00000757   (duration 19910.600)
628845589.940 → 628865500.540   qualimap_00000121   (duration 19910.600)
628865500.540 → 628872522.340   bismark_report_00000474   (duration 7021.800)
628872522.340 → 628873984.740   get_software_versions_00000154   (duration 1462.400)
628873984.740 → 628875447.140   bismark_summary_00000038   (duration 1462.400)
628875447.140 → 628876909.540   get_software_versions_00000250   (duration 1462.400)
637647749.580 → 637649211.980   preseq_00000131   (duration 1462.400)
637649211.980 → 637650674.380   get_software_versions_00000238   (duration 1462.400)
637650674.380 → 637652136.780 

In [38]:
merged.to_csv("merged.csv")

In [39]:
import pandas as pd
import numpy as np

# === CONFIG: update file names as needed ===
merged_file = "merged.csv"          # your merged dynamic + static dataframe
# If merged.csv doesn't exist, you can use separate dynamic/static files:
dyn_file = "df_wide2.csv"
stat_file = "stat_methylseq2.csv"

# === 1. Load merged dataframe ===
try:
    df = pd.read_csv(merged_file)
    print(f"Loaded merged file: {merged_file}")
except FileNotFoundError:
    print(f"merged file not found, attempting to merge {dyn_file} + {stat_file}")
    dfdyn = pd.read_csv(dyn_file)
    dfstat = pd.read_csv(stat_file)
    # rename to canonical column names
    dfdyn = dfdyn.rename(columns={
        "start_time": "start_time_dyn",
        "end_time": "end_time_dyn",
        "duration": "duration_dyn",
        "proc": "processor_dyn",
        "processor": "processor_dyn"
    })
    dfstat = dfstat.rename(columns={
        "start_time": "start_time_stat",
        "end_time": "end_time_stat",
        "duration": "duration_stat",
        "processor": "processor_stat",
        "proc": "processor_stat"
    })
    df = dfdyn.merge(dfstat, on="vertex", suffixes=("_dyn","_stat"))
    print("Merged dynamic + static into dataframe.")

# === 2. Sanity check columns ===
needed = ["vertex",
          "start_time_dyn","end_time_dyn","duration_dyn","processor_dyn",
          "start_time_stat","end_time_stat","duration_stat","processor_stat"]
print("Column presence:")
for c in needed:
    print(f"  {c}: {'OK' if c in df.columns else 'MISSING'}")

# === 3. Convert numeric columns ===
for c in ["start_time_dyn","end_time_dyn","duration_dyn","start_time_stat","end_time_stat","duration_stat"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["processor_dyn","processor_stat"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce", downcast='integer')

# === 4. Compute differences & flags ===
df["dur_diff_abs"] = df["duration_stat"] - df["duration_dyn"]
df["dur_ratio"] = df["duration_stat"] / df["duration_dyn"].replace(0,np.nan)
df["static_worse"] = df["dur_diff_abs"] > 0
df["same_processor"] = df["processor_dyn"] == df["processor_stat"]

# === 5. Summary ===
total = len(df)
total_worse = df["static_worse"].sum()
same_proc_worse = df[df["static_worse"] & df["same_processor"]].shape[0]
print(f"Total tasks: {total}; static worse: {total_worse} ({total_worse/total:.2%})")
print(f"Static worse on same processor: {same_proc_worse} ({same_proc_worse/total:.2%})")

# === 6. Per-processor aggregates ===
per_proc = []
for proc in sorted(df["processor_dyn"].dropna().unique()):
    g = df[df["processor_dyn"] == proc]
    n = len(g)
    n_worse = g["static_worse"].sum()
    median_ratio = g["dur_ratio"].median()
    median_abs = g["dur_diff_abs"].median()
    per_proc.append((proc, n, n_worse, median_ratio, median_abs))

per_proc_df = pd.DataFrame(per_proc, columns=["processor","count","count_static_worse","median_dur_ratio","median_abs_diff"])
per_proc_df = per_proc_df.sort_values("median_dur_ratio", ascending=False)
print("\nPer-processor summary (sorted by median ratio):")
display(per_proc_df)

# === 7. Top offenders ===
top_abs = df[df["dur_diff_abs"]>0].sort_values("dur_diff_abs", ascending=False)
top_rel = df[df["dur_ratio"]>1].sort_values("dur_ratio", ascending=False)

print("\nTop 20 tasks by absolute static>dynamic duration difference:")
display(top_abs.head(20)[["vertex","processor_dyn","processor_stat","duration_dyn","duration_stat","dur_diff_abs","dur_ratio"]])

print("\nTop 20 tasks by relative ratio (static/dynamic):")
display(top_rel.head(20)[["vertex","processor_dyn","processor_stat","duration_dyn","duration_stat","dur_ratio","dur_diff_abs"]])

# === 8. Optional: per-processor top 5 worst tasks ===
for proc in sorted(df["processor_dyn"].dropna().unique()):
    g = df[(df["processor_dyn"]==proc) & (df["dur_diff_abs"]>0)].sort_values("dur_diff_abs", ascending=False).head(5)
    if len(g):
        print(f"\nProcessor {int(proc)}: top {len(g)} static-worse tasks:")
        display(g[["vertex","duration_dyn","duration_stat","dur_diff_abs","dur_ratio"]])


Loaded merged file: merged.csv
Column presence:
  vertex: OK
  start_time_dyn: OK
  end_time_dyn: OK
  duration_dyn: OK
  processor_dyn: OK
  start_time_stat: OK
  end_time_stat: OK
  duration_stat: OK
  processor_stat: OK
Total tasks: 992; static worse: 604 (60.89%)
Static worse on same processor: 19 (1.92%)

Per-processor summary (sorted by median ratio):


,processor,count,count_static_worse,median_dur_ratio,median_abs_diff
10,10,204,175,4.000000,7.921333e+02
8,8,65,47,4.000000,5.484000e+02
9,9,85,67,2.666667,7.921333e+02
7,7,24,18,2.666667,5.484000e+02
6,6,26,16,2.666667,3.046667e+02
18,18,6,4,2.250000,6.093333e+02
27,27,5,4,2.000000,7.312000e+02
21,21,6,4,1.750000,3.656000e+02
28,28,6,4,1.666667,4.874667e+02
26,26,4,3,1.666667,4.874667e+02



Top 20 tasks by absolute static>dynamic duration difference:


,vertex,processor_dyn,processor_stat,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
482,fastqc_00000848,9,4,1.093625e+06,8.749000e+06,7.655375e+06,8.000000
438,fastqc_00000320,11,2,1.093625e+06,8.749000e+06,7.655375e+06,8.000000
437,fastqc_00000308,8,0,1.093625e+06,8.749000e+06,7.655375e+06,8.000000
447,fastqc_00000428,18,1,2.916333e+06,8.749000e+06,5.832666e+06,3.000000
446,fastqc_00000416,21,3,2.916333e+06,8.749000e+06,5.832666e+06,3.000000
432,fastqc_00000248,11,13,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
485,fastqc_00000884,8,12,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
435,fastqc_00000284,10,17,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
434,fastqc_00000272,9,15,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
433,fastqc_00000260,6,14,1.093625e+06,5.832666e+06,4.739041e+06,5.333333



Top 20 tasks by relative ratio (static/dynamic):


...

,vertex,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
494,fastqc_00000992,1.093625e+06,2.916333e+06,1.822708e+06,2.666667
33,bismark_align_00000435,6.187713e+06,7.599916e+06,1.412203e+06,1.228227
63,bismark_align_00000795,6.187713e+06,7.599916e+06,1.412203e+06,1.228227
23,bismark_align_00000291,6.187713e+06,7.599916e+06,1.412203e+06,1.228227
163,bismark_methXtract_00000010,7.639925e+03,2.037313e+04,1.273321e+04,2.666667



Processor 34: top 5 static-worse tasks:


,vertex,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
25,bismark_align_00000315,6187713.025,7.599918e+06,1.412205e+06,1.228227
69,bismark_align_00000867,6187713.025,7.599914e+06,1.412201e+06,1.228227
41,bismark_align_00000531,6187713.025,7.599914e+06,1.412201e+06,1.228227
165,bismark_methXtract_00000028,7639.925,6.111940e+04,5.347947e+04,8.000000
248,bismark_report_00000030,877.725,2.340600e+03,1.462875e+03,2.666667



Processor 35: top 5 static-worse tasks:


,vertex,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
61,bismark_align_00000771,6187713.025,7.599918e+06,1.412205e+06,1.228227
37,bismark_align_00000483,6187713.025,7.599914e+06,1.412201e+06,1.228227
22,bismark_align_00000267,6187713.025,7.599914e+06,1.412201e+06,1.228227
831,qualimap_00000049,2488.825,6.636867e+03,4.148042e+03,2.666667
829,qualimap_00000025,2488.825,6.636867e+03,4.148042e+03,2.666667


In [54]:
fname= "./dynamic_results_atacseq_200.txt"
dyn_atacseq = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="warn")
#print(dyn_atacseq.head())

# Pivot start/finish times
df_wide = (
    dyn_atacseq.pivot(index="vertex", columns="startOrFinish", values="time")
      .rename(columns={"start": "start_time", "finish": "end_time"})
      .reset_index()
)

# Add processor where start happens
df_wide = df_wide.merge(
    dyn_atacseq[dyn_atacseq["startOrFinish"] == "start"][["vertex", "proc"]], 
    on="vertex"
)


df_wide["duration"] = df_wide["end_time"] - df_wide["start_time"]
df_wide2 = df_wide.rename(columns={"proc": "processor",
                                 "duration": "duration"})

#print(df_wide.head())
df_wide2.to_csv("dynamic_schedule_atac_200.csv")

fname= "./static_results_atacseq_200.txt"
stat_atacseq = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="warn")

stat_atacseq["duration"] = stat_atacseq["finish"] - stat_atacseq["start"]

stat_atacseq2 = stat_atacseq.rename(columns={"start": "start_time",                
                                 "finish": "end_time", "proc": "processor"})



#print(stat_methylseq.head())
stat_atacseq2.to_csv("static_schedule_atac_200.csv")

merged = df_wide2.merge(
    stat_atacseq2,
    on="vertex",
    suffixes=("_dyn", "_stat")
)
print(merged.head())


# Rows where duration differs
diff = merged[ merged["duration_dyn"] != merged["duration_stat"] ]

# Sort by dynamic start time
diff = diff.sort_values(by="start_time_dyn")

res = diff[[
    "vertex",
    "start_time_dyn",
    "duration_dyn", "processor_dyn",
    "start_time_stat",
    "duration_stat", "processor_stat"
]]

#print(res)


#res.to_csv("dyn_and_stat_ATAC200_compared.csv", index=False)

same_proc_diff_duration = merged[
    (merged["duration_dyn"] != merged["duration_stat"]) &
    (merged["processor_dyn"] == merged["processor_stat"])
]

# Sort by start time (dynamic start)
same_proc_diff_duration = same_proc_diff_duration.sort_values(by="start_time_dyn")

print(same_proc_diff_duration[[
    "vertex",
    "processor_dyn",
    "duration_dyn",
    "duration_stat",
    "start_time_dyn",
    "start_time_stat"
]])

same_proc_diff_duration[[
    "vertex",
    "processor_dyn",
    "duration_dyn",
    "duration_stat",
    "start_time_dyn",
    "start_time_stat"
]].to_csv("same_proc_diff_durationATAC.csv", index=False)
merged.to_csv("mergedAtacseq.csv")

               vertex  end_time_dyn  start_time_dyn  processor_dyn  \
0  BWA_INDEX_00000005   154726.3000          0.0000             22   
1  BWA_INDEX_00000069   116044.7250      58022.3625              8   
2  BWA_INDEX_00000108   116044.7250      58022.3625             32   
3  BWA_INDEX_00000147    58022.3625          0.0000              9   
4  BWA_INDEX_00000186    58022.3625          0.0000             32   

   duration_dyn  start_time_stat  end_time_stat  processor_stat  duration_stat  
0   154726.3000              0.0     58022.3625              11     58022.3625  
1    58022.3625              0.0     58022.3625               8     58022.3625  
2    58022.3625              0.0     58022.3625              10     58022.3625  
3    58022.3625              0.0     58022.3625              30     58022.3625  
4    58022.3625              0.0     58022.3625               9     58022.3625  
                                   vertex  processor_dyn  duration_dyn  \
11                 

In [53]:
import pandas as pd
import numpy as np

# === CONFIG: update file names as needed ===
merged_file = "merged.csv"          # your merged dynamic + static dataframe
# If merged.csv doesn't exist, you can use separate dynamic/static files:
dyn_file = "df_wide2.csv"
stat_file = "stat_methylseq2.csv"

# === 1. Load merged dataframe ===
try:
    df = pd.read_csv(merged_file)
    print(f"Loaded merged file: {merged_file}")
except FileNotFoundError:
    print(f"merged file not found, attempting to merge {dyn_file} + {stat_file}")
    dfdyn = pd.read_csv(dyn_file)
    dfstat = pd.read_csv(stat_file)
    # rename to canonical column names
    dfdyn = dfdyn.rename(columns={
        "start_time": "start_time_dyn",
        "end_time": "end_time_dyn",
        "duration": "duration_dyn",
        "proc": "processor_dyn",
        "processor": "processor_dyn"
    })
    dfstat = dfstat.rename(columns={
        "start_time": "start_time_stat",
        "end_time": "end_time_stat",
        "duration": "duration_stat",
        "processor": "processor_stat",
        "proc": "processor_stat"
    })
    df = dfdyn.merge(dfstat, on="vertex", suffixes=("_dyn","_stat"))
    print("Merged dynamic + static into dataframe.")

# === 2. Sanity check columns ===
needed = ["vertex",
          "start_time_dyn","end_time_dyn","duration_dyn","processor_dyn",
          "start_time_stat","end_time_stat","duration_stat","processor_stat"]
print("Column presence:")
for c in needed:
    print(f"  {c}: {'OK' if c in df.columns else 'MISSING'}")

# === 3. Convert numeric columns ===
for c in ["start_time_dyn","end_time_dyn","duration_dyn","start_time_stat","end_time_stat","duration_stat"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["processor_dyn","processor_stat"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce", downcast='integer')

# === 4. Compute differences & flags ===
df["dur_diff_abs"] = df["duration_stat"] - df["duration_dyn"]
df["dur_ratio"] = df["duration_stat"] / df["duration_dyn"].replace(0,np.nan)
df["static_worse"] = df["dur_diff_abs"] > 0
df["same_processor"] = df["processor_dyn"] == df["processor_stat"]

# === 5. Summary ===
total = len(df)
total_worse = df["static_worse"].sum()
same_proc_worse = df[df["static_worse"] & df["same_processor"]].shape[0]
print(f"Total tasks: {total}; static worse: {total_worse} ({total_worse/total:.2%})")
print(f"Static worse on same processor: {same_proc_worse} ({same_proc_worse/total:.2%})")

# === 6. Per-processor aggregates ===
per_proc = []
for proc in sorted(df["processor_dyn"].dropna().unique()):
    g = df[df["processor_dyn"] == proc]
    n = len(g)
    n_worse = g["static_worse"].sum()
    median_ratio = g["dur_ratio"].median()
    median_abs = g["dur_diff_abs"].median()
    per_proc.append((proc, n, n_worse, median_ratio, median_abs))

per_proc_df = pd.DataFrame(per_proc, columns=["processor","count","count_static_worse","median_dur_ratio","median_abs_diff"])
per_proc_df = per_proc_df.sort_values("median_dur_ratio", ascending=False)
print("\nPer-processor summary (sorted by median ratio):")
display(per_proc_df)

# === 7. Top offenders ===
top_abs = df[df["dur_diff_abs"]>0].sort_values("dur_diff_abs", ascending=False)
top_rel = df[df["dur_ratio"]>1].sort_values("dur_ratio", ascending=False)

print("\nTop 20 tasks by absolute static>dynamic duration difference:")
display(top_abs.head(20)[["vertex","processor_dyn","processor_stat","duration_dyn","duration_stat","dur_diff_abs","dur_ratio"]])

print("\nTop 20 tasks by relative ratio (static/dynamic):")
display(top_rel.head(20)[["vertex","processor_dyn","processor_stat","duration_dyn","duration_stat","dur_ratio","dur_diff_abs"]])

# === 8. Optional: per-processor top 5 worst tasks ===
for proc in sorted(df["processor_dyn"].dropna().unique()):
    g = df[(df["processor_dyn"]==proc) & (df["dur_diff_abs"]>0)].sort_values("dur_diff_abs", ascending=False).head(5)
    if len(g):
        print(f"\nProcessor {int(proc)}: top {len(g)} static-worse tasks:")
        display(g[["vertex","duration_dyn","duration_stat","dur_diff_abs","dur_ratio"]])

Loaded merged file: merged.csv
Column presence:
  vertex: OK
  start_time_dyn: OK
  end_time_dyn: OK
  duration_dyn: OK
  processor_dyn: OK
  start_time_stat: OK
  end_time_stat: OK
  duration_stat: OK
  processor_stat: OK
Total tasks: 992; static worse: 604 (60.89%)
Static worse on same processor: 19 (1.92%)

Per-processor summary (sorted by median ratio):


,processor,count,count_static_worse,median_dur_ratio,median_abs_diff
10,10,204,175,4.000000,7.921333e+02
8,8,65,47,4.000000,5.484000e+02
9,9,85,67,2.666667,7.921333e+02
7,7,24,18,2.666667,5.484000e+02
6,6,26,16,2.666667,3.046667e+02
18,18,6,4,2.250000,6.093333e+02
27,27,5,4,2.000000,7.312000e+02
21,21,6,4,1.750000,3.656000e+02
28,28,6,4,1.666667,4.874667e+02
26,26,4,3,1.666667,4.874667e+02



Top 20 tasks by absolute static>dynamic duration difference:


,vertex,processor_dyn,processor_stat,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
482,fastqc_00000848,9,4,1.093625e+06,8.749000e+06,7.655375e+06,8.000000
438,fastqc_00000320,11,2,1.093625e+06,8.749000e+06,7.655375e+06,8.000000
437,fastqc_00000308,8,0,1.093625e+06,8.749000e+06,7.655375e+06,8.000000
447,fastqc_00000428,18,1,2.916333e+06,8.749000e+06,5.832666e+06,3.000000
446,fastqc_00000416,21,3,2.916333e+06,8.749000e+06,5.832666e+06,3.000000
432,fastqc_00000248,11,13,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
485,fastqc_00000884,8,12,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
435,fastqc_00000284,10,17,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
434,fastqc_00000272,9,15,1.093625e+06,5.832666e+06,4.739041e+06,5.333333
433,fastqc_00000260,6,14,1.093625e+06,5.832666e+06,4.739041e+06,5.333333



Top 20 tasks by relative ratio (static/dynamic):


...

,vertex,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
494,fastqc_00000992,1.093625e+06,2.916333e+06,1.822708e+06,2.666667
33,bismark_align_00000435,6.187713e+06,7.599916e+06,1.412203e+06,1.228227
63,bismark_align_00000795,6.187713e+06,7.599916e+06,1.412203e+06,1.228227
23,bismark_align_00000291,6.187713e+06,7.599916e+06,1.412203e+06,1.228227
163,bismark_methXtract_00000010,7.639925e+03,2.037313e+04,1.273321e+04,2.666667



Processor 34: top 5 static-worse tasks:


,vertex,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
25,bismark_align_00000315,6187713.025,7.599918e+06,1.412205e+06,1.228227
69,bismark_align_00000867,6187713.025,7.599914e+06,1.412201e+06,1.228227
41,bismark_align_00000531,6187713.025,7.599914e+06,1.412201e+06,1.228227
165,bismark_methXtract_00000028,7639.925,6.111940e+04,5.347947e+04,8.000000
248,bismark_report_00000030,877.725,2.340600e+03,1.462875e+03,2.666667



Processor 35: top 5 static-worse tasks:


,vertex,duration_dyn,duration_stat,dur_diff_abs,dur_ratio
61,bismark_align_00000771,6187713.025,7.599918e+06,1.412205e+06,1.228227
37,bismark_align_00000483,6187713.025,7.599914e+06,1.412201e+06,1.228227
22,bismark_align_00000267,6187713.025,7.599914e+06,1.412201e+06,1.228227
831,qualimap_00000049,2488.825,6.636867e+03,4.148042e+03,2.666667
829,qualimap_00000025,2488.825,6.636867e+03,4.148042e+03,2.666667


In [4]:
path = "./results-26-12/merged/*.txt"
#print(path)

patterndevs = r'^(BASE|A\d+)-(ndev)'

dfs26=read_dfs(path,patterndevs, 2)

path = "./results-18-12/merged/*.txt"
#print(path)

patterndevs = r'^(BASE|A\d+)-(ndev)'

dfs18=read_dfs(path,patterndevs, 2)
#print(dfs18)

{('A1', 'ndev'):      algo_nr              wf_name     inp_size    dur_alg1          ms_1  \
0          1    atacseq_30000.dot  14091675276   27.869306  2.208628e+09   
1          1    chipseq_15000.dot  41366257414   10.057338  3.608378e+09   
2          1  methylseq_15000.dot   6761426956  108.373746  5.701489e+08   
3          1    atacseq_15000.dot   3908761308   13.129148  3.323044e+08   
4          1                eager  19075314980    0.034273  7.610497e+06   
..       ...                  ...          ...         ...           ...   
259        1                eager  19132169434    0.034361  7.614341e+06   
260        1  methylseq_10000.dot  17027257018   29.445993  1.142015e+09   
261        1        eager_200.dot   8330435694    0.126112  4.959908e+06   
262        1              atacseq  11809629756    0.009802  3.111618e+06   
263        1    chipseq_10000.dot   4807293396    7.670184  3.880994e+08   

          ms_perc   dur_alg2          ms_2  
0    2.324460e+09  46.129

In [23]:

def compare_equal_2dates(dfs1, dfs2):
  
    comparison_results = {}

    for key in dfs1.keys():
        if key not in dfs2:
            print(f"Key {key} missing in dfs26, skipping")
            continue

        df1 = dfs1[key]
        df2 = dfs2[key]

        # merge by workflow and input size
        merged = df1.merge(
            df2,
            on=["wf_name", "inp_size"],
            suffixes=("_1", "_2")
        )
       # print(merged)

        # compare equality
        merged["equal_ms_1"] = merged["ms_1_1"] == merged["ms_1_2"]
        merged["equal_ms_2"] = merged["ms_2_1"] == merged["ms_2_2"]

        # store for later inspection
        comparison_results[key] = merged

        # quick summary
        print(f"\nResults for {key}:")
        print("ms_1 equal:", merged["equal_ms_1"].mean(), "fraction of cases")
        print("ms_2 equal:", merged["equal_ms_2"].mean(), "fraction of cases")


In [9]:
path = "./results-26-12/merged/*.txt"
#print(path)

patterndevs = r'^(BASE|A\d+)-(ndev)'

dfs26=read_dfs(path,patterndevs, 2)

path = "./results-18-12/merged/*.txt"
#print(path)

patterndevs = r'^(BASE|A\d+)-(ndev)'

dfs18=read_dfs(path,patterndevs, 2)
#print(dfs18)

compare_equal_2dates(dfs18, dfs26)


Results for ('A1', 'ndev'):
ms_1 equal: 1.0 fraction of cases
ms_2 equal: 1.0 fraction of cases

Results for ('A2', 'ndev'):
ms_1 equal: 1.0 fraction of cases
ms_2 equal: 1.0 fraction of cases

Results for ('A3', 'ndev'):
ms_1 equal: 1.0 fraction of cases
ms_2 equal: 1.0 fraction of cases

Results for ('BASE', 'ndev'):
ms_1 equal: 1.0 fraction of cases
ms_2 equal: 1.0 fraction of cases


In [24]:
path = "./results-26-12/merged/*.txt"
#print(path)

patterndevs = r'^(BASE|A\d+)-(bdev)'

dfs26=read_dfs(path,patterndevs, 2)
#print(dfs26)

path = "./results-18-12/merged/*.txt"
#print(path)

patterndevs = r'^(BASE|A\d+)-(50dev)'

dfs18=read_dfs(path,patterndevs, 2)
dfs18 = {
    (k1, "bdev" if k2 == "50dev" else k2): df
    for (k1, k2), df in dfs18.items()
}
#print(dfs18.keys())

compare_equal_2dates(dfs18, dfs26)


Results for ('BASE', 'bdev'):
ms_1 equal: 0.0 fraction of cases
ms_2 equal: 0.0 fraction of cases

Results for ('A1', 'bdev'):
ms_1 equal: 0.0 fraction of cases
ms_2 equal: 0.0 fraction of cases

Results for ('A2', 'bdev'):
ms_1 equal: 0.0 fraction of cases
ms_2 equal: 0.0 fraction of cases

Results for ('A3', 'bdev'):
ms_1 equal: 0.0 fraction of cases
ms_2 equal: 0.0 fraction of cases


In [16]:
out_stat= '''
CHECK_DESIGN_00000001 0 30 0 1676.96166666667 1676.96166666667
CHECK_DESIGN_00000049 0 31 0 1676.96166666667 1676.96166666667
CHECK_DESIGN_00000074 0 32 0 1676.96166666667 1676.96166666667
CHECK_DESIGN_00000099 0 33 0 1676.96166666667 1676.96166666667
CHECK_DESIGN_00000124 0 34 0 1676.96166666667 1676.96166666667
CHECK_DESIGN_00000149 0 35 0 1676.96166666667 1676.96166666667
CHECK_DESIGN_00000174 0 30 1676.96166666667 3353.92333333333 1676.96166666667
TRIMGALORE_00000002 0 30 3353.92333333333 7865.3925 4511.46916666667
TRIMGALORE_00000030 0 31 1676.96166666667 6188.43083333333 4511.46916666667
TRIMGALORE_00000055 0 32 1676.96166666667 6188.43083333333 4511.46916666667
TRIMGALORE_00000080 0 33 1676.96166666667 6188.43083333333 4511.46916666667
TRIMGALORE_00000105 0 34 1676.96166666667 6188.43083333333 4511.46916666667
TRIMGALORE_00000130 0 35 1676.96166666667 6188.43083333333 4511.46916666667
TRIMGALORE_00000155 0 30 7865.3925 12376.8616666667 4511.46916666667
BWA_MEM_00000009 1 30 12376.8616666667 614541.05117905 602164.189512383
BWA_MEM_00000038 0 31 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000063 0 32 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000088 0 33 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000113 0 34 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000138 0 35 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000163 1 30 614541.05117905 715066.86300286 100525.81182381
SORT_BAM_00000011 0 30 715066.86300286 717761.574669527 2694.71166666667
SORT_BAM_00000040 0 31 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000065 0 32 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000090 0 33 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000115 0 34 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000140 0 35 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000165 0 30 717761.574669527 720456.286336193 2694.71166666667
MERGED_BAM_00000012 1 30 720456.286336193 731192.467893983 10736.1815577899
MERGED_BAM_00000046 0 31 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_00000071 0 32 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_00000096 0 33 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_00000121 0 34 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_00000146 0 35 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_00000171 1 30 731192.467893983 819163.924288843 87971.4563948598
MAKE_GENOME_FILTER_00000020 0 6 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000031 0 7 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000056 0 8 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000081 0 9 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000106 0 10 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000131 0 11 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000156 0 6 419.240416666667 838.480833333333 419.240416666667
MERGED_BAM_FILTER_00000005 2 30 956967.916531918 962017.080698584 5049.16416666668
MERGED_BAM_FILTER_00000037 0 31 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_FILTER_00000062 0 32 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_FILTER_00000087 0 33 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_FILTER_00000112 0 34 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_FILTER_00000137 0 35 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_FILTER_00000162 2 31 1537705.4652334 1542754.62940006 5049.16416666657
MACS2_00000006 0 30 962017.080698584 963694.042365251 1676.96166666667
MACS2_00000028 0 18 1043528.65138124 1044646.62582569 1117.97444444441
MACS2_00000053 0 32 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000078 0 33 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000103 0 34 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000128 0 35 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000153 0 31 1542754.62940006 1544431.59106673 1676.96166666667
BIGWIG_00000021 0 30 963694.042365251 968375.981531918 4681.93916666671
BIGWIG_00000039 0 30 1389781.02992408 1394462.96909075 4681.93916666671
BIGWIG_00000064 0 32 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000089 0 33 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000114 0 34 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000139 0 35 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000164 0 31 1544431.59106673 1549113.5302334 4681.93916666671
CONSENSUS_PEAKS_00000008 0 18 1334993.59222667 1336111.56667111 1117.97444444452
CONSENSUS_PEAKS_00000047 0 19 1132895.0438273 1134013.01827175 1117.97444444452
CONSENSUS_PEAKS_00000072 0 32 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_00000097 0 33 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_00000122 0 34 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_00000147 0 35 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_00000172 0 31 1549113.5302334 1550790.49190006 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000007 0 19 1379004.5876125 1380122.56205695 1117.97444444452
CONSENSUS_PEAKS_COUNTS_00000026 0 20 1282363.17566654 1283481.15011099 1117.97444444452
CONSENSUS_PEAKS_COUNTS_00000051 0 32 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000076 0 33 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000101 0 34 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000126 0 35 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000151 0 31 1550790.49190006 1552467.45356673 1676.96166666667
MACS2_ANNOTATE_00000023 0 30 1394462.96909075 1396139.93075741 1676.96166666667
MACS2_ANNOTATE_00000034 0 18 1336111.56667111 1337229.54111556 1117.97444444452
MACS2_ANNOTATE_00000059 0 32 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000084 0 33 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000109 0 34 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000134 0 35 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000159 0 31 1552467.45356673 1554144.4152334 1676.96166666667
PHANTOMPEAKQUALTOOLS_00000014 0 30 1396139.93075741 1403977.32534075 7837.39458333328
PHANTOMPEAKQUALTOOLS_00000027 0 31 1554144.4152334 1561981.80981673 7837.39458333328
PHANTOMPEAKQUALTOOLS_00000052 0 32 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000077 0 33 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000102 0 34 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000127 0 35 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000152 0 31 1561981.80981673 1569819.20440006 7837.39458333328
PLOTPROFILE_00000016 0 30 1403977.32534075 1413302.18992408 9324.86458333326
PLOTPROFILE_00000044 0 30 1413302.18992408 1422627.05450741 9324.86458333326
PLOTPROFILE_00000069 0 32 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000094 0 33 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000119 0 34 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000144 0 35 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000169 0 31 1569819.20440006 1579144.0689834 9324.86458333326
PRESEQ_00000013 0 30 1422627.05450741 1426489.69659075 3862.64208333334
PRESEQ_00000032 0 31 1579144.0689834 1583006.71106673 3862.64208333334
PRESEQ_00000057 0 32 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000082 0 33 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000107 0 34 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000132 0 35 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000157 0 30 1426489.69659075 1430352.33867408 3862.64208333334
FASTQC_00000003 0 30 1430352.33867408 1432593.36742408 2241.02875000006
FASTQC_00000042 0 32 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000067 0 33 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000092 0 34 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000117 0 35 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000142 0 35 1915111.10776269 1917352.13651269 2241.02875000006
FASTQC_00000167 0 12 2307688.0300654 2310676.06839873 2988.03833333356
CONSENSUS_PEAKS_DESEQ2_00000018 0 19 1380122.56205695 1381240.53650139 1117.97444444452
CONSENSUS_PEAKS_DESEQ2_00000029 0 20 1283481.15011099 1284599.12455543 1117.97444444452
CONSENSUS_PEAKS_DESEQ2_00000054 0 21 1043539.37286687 1044657.34731131 1117.97444444441
CONSENSUS_PEAKS_DESEQ2_00000079 0 22 1043539.37286687 1044657.34731131 1117.97444444441
CONSENSUS_PEAKS_DESEQ2_00000104 0 23 1043539.37286687 1044657.34731131 1117.97444444441
CONSENSUS_PEAKS_DESEQ2_00000129 0 24 139412.084251617 141089.045918284 1676.96166666667
CONSENSUS_PEAKS_DESEQ2_00000154 0 31 1583006.71106673 1584683.6727334 1676.96166666667
PICARD_METRICS_00000015 0 30 1432593.36742408 1434270.32909075 1676.96166666667
PICARD_METRICS_00000033 0 31 1584683.6727334 1586360.63440006 1676.96166666667
PICARD_METRICS_00000058 0 32 1915111.10776269 1916788.06942935 1676.96166666667
PICARD_METRICS_00000083 0 33 1915111.10776269 1916788.06942935 1676.96166666667
PICARD_METRICS_00000108 0 34 1915111.10776269 1916788.06942935 1676.96166666667
PICARD_METRICS_00000133 0 35 1917352.13651269 1919029.09817935 1676.96166666667
PICARD_METRICS_00000158 0 31 1586360.63440006 1588037.59606673 1676.96166666667
PLOTFINGERPRINT_00000004 0 18 1423048.24853847 1424166.22298292 1117.97444444452
MACS2_QC_00000017 0 30 1434270.32909075 1435947.29075741 1676.96166666667
get_software_versions_00000019 0 7 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000035 0 20 1402354.62773535 1403472.60217979 1117.97444444452
PLOTFINGERPRINT_00000050 0 21 1326406.88194879 1327524.85639324 1117.97444444452
get_software_versions_00000045 0 8 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000060 0 24 1065555.77544437 1067232.73711104 1676.96166666667
PLOTFINGERPRINT_00000075 0 22 1131594.02796832 1132712.00241276 1117.97444444452
get_software_versions_00000070 0 9 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000085 0 25 1065555.77544437 1067232.73711104 1676.96166666667
PLOTFINGERPRINT_00000100 0 23 1131594.02796832 1132712.00241276 1117.97444444452
get_software_versions_00000095 0 10 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000110 0 26 1065555.77544437 1067232.73711104 1676.96166666667
PLOTFINGERPRINT_00000125 0 24 1131594.02796832 1133270.98963498 1676.96166666667
get_software_versions_00000120 0 11 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000135 0 27 161428.48682912 163105.448495787 1676.96166666667
PLOTFINGERPRINT_00000150 0 27 227466.739353067 229143.701019734 1676.96166666667
get_software_versions_00000145 0 6 838.480833333333 1257.72125 419.240416666667
MACS2_QC_00000160 0 31 1588037.59606673 1589714.5577334 1676.96166666667
PLOTFINGERPRINT_00000175 0 31 1589714.5577334 1591391.51940006 1676.96166666667
get_software_versions_00000170 0 7 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000010 0 30 1435947.29075741 1437624.25242408 1676.96166666667
IGV_00000022 0 30 1437624.25242408 1439301.21409075 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000024 0 18 1424166.22298292 1425284.19742736 1117.97444444452
output_documentation_00000025 0 8 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000041 0 8 1915117.79632754 1915537.03674421 419.240416666726
IGV_00000036 0 30 1439301.21409075 1440978.17575741 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000043 0 19 1381240.53650139 1382358.51094583 1117.97444444452
output_documentation_00000048 0 9 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000066 0 32 1916788.06942935 1918465.03109602 1676.96166666667
IGV_00000061 0 32 1918465.03109602 1920141.99276269 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000068 0 32 1920141.99276269 1921818.95442935 1676.96166666667
output_documentation_00000073 0 10 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000091 0 33 1916788.06942935 1918465.03109602 1676.96166666667
IGV_00000086 0 33 1918465.03109602 1920141.99276269 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000093 0 33 1920141.99276269 1921818.95442935 1676.96166666667
output_documentation_00000098 0 11 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000116 0 34 1916788.06942935 1918465.03109602 1676.96166666667
IGV_00000111 0 34 1918465.03109602 1920141.99276269 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000118 0 34 1920141.99276269 1921818.95442935 1676.96166666667
output_documentation_00000123 0 28 0 1676.96166666667 1676.96166666667
MULTIQC_00000141 0 35 1919029.09817935 1920706.05984602 1676.96166666667
IGV_00000136 0 35 1920706.05984602 1922383.02151269 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000143 0 35 1922383.02151269 1924059.98317935 1676.96166666667
output_documentation_00000148 0 29 0 1676.96166666667 1676.96166666667
MULTIQC_00000166 0 7 2310683.89185791 2311103.13227457 419.240416666493
IGV_00000161 0 31 1591391.51940006 1593068.48106673 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000168 0 31 1593068.48106673 1594745.4427334 1676.96166666667
output_documentation_00000173 0 6 1257.72125 1676.96166666667 419.240416666667
'''

out_dyn = '''
CHECK_DESIGN_00000001 0 30 0 1676.96166666667 1676.96166666667
TRIMGALORE_00000002 0 30 1676.96166666667 6188.43083333333 4511.46916666667
CHECK_DESIGN_00000049 0 31 0 1676.96166666667 1676.96166666667
TRIMGALORE_00000030 0 31 1676.96166666667 6188.43083333333 4511.46916666667
CHECK_DESIGN_00000074 0 32 0 1676.96166666667 1676.96166666667
TRIMGALORE_00000055 0 32 1676.96166666667 6188.43083333333 4511.46916666667
CHECK_DESIGN_00000099 0 33 0 1676.96166666667 1676.96166666667
TRIMGALORE_00000080 0 33 1676.96166666667 6188.43083333333 4511.46916666667
CHECK_DESIGN_00000124 0 34 0 1676.96166666667 1676.96166666667
TRIMGALORE_00000105 0 34 1676.96166666667 6188.43083333333 4511.46916666667
CHECK_DESIGN_00000149 0 35 0 1676.96166666667 1676.96166666667
TRIMGALORE_00000130 0 35 1676.96166666667 6188.43083333333 4511.46916666667
CHECK_DESIGN_00000174 0 30 6188.43083333333 7865.3925 1676.96166666667
TRIMGALORE_00000155 0 30 7865.3925 12376.8616666667 4511.46916666667
BWA_MEM_00000009 1 30 12376.8616666667 614541.05117905 602164.189512384
BWA_MEM_00000038 0 31 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000063 0 32 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000088 0 33 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000113 0 34 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000138 0 35 6188.43083333333 23942.1258333333 17753.695
BWA_MEM_00000163 1 30 614541.05117905 715066.863002861 100525.81182381
MAKE_GENOME_FILTER_00000031 0 6 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000056 0 7 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000081 0 8 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000106 0 9 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000131 0 10 0 419.240416666667 419.240416666667
MAKE_GENOME_FILTER_00000156 0 11 0 419.240416666667 419.240416666667
SORT_BAM_00000011 0 30 715066.863002861 717761.574669527 2694.71166666667
SORT_BAM_00000040 0 31 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000065 0 32 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000090 0 33 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000115 0 34 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000140 0 35 23942.1258333333 26636.8375 2694.71166666667
SORT_BAM_00000165 0 30 717761.574669527 720456.286336194 2694.71166666667
MERGED_BAM_00000012 1 30 720456.286336194 731192.467893984 10736.18155779
MERGED_BAM_00000046 0 31 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_FILTER_00000037 0 31 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_00000071 0 32 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_FILTER_00000062 0 32 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_00000096 0 33 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_FILTER_00000087 0 33 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_00000121 0 34 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_FILTER_00000112 0 34 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_00000146 0 35 26636.8375 36401.6779166667 9764.84041666667
MERGED_BAM_FILTER_00000137 0 35 36401.6779166667 41450.8420833333 5049.16416666667
MERGED_BAM_00000171 1 30 731192.467893984 819163.924288844 87971.4563948599
MERGED_BAM_FILTER_00000162 2 30 953613.993198584 966528.549865251 12914.5566666666
MACS2_00000028 0 31 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000053 0 32 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000078 0 33 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000103 0 34 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000128 0 35 41450.8420833333 43127.80375 1676.96166666667
MACS2_00000153 0 30 966528.549865251 968205.511531918 1676.96166666667
BIGWIG_00000039 0 31 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000064 0 32 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000089 0 33 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000114 0 34 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000139 0 35 43127.80375 47809.7429166667 4681.93916666666
BIGWIG_00000164 0 30 968205.511531918 972887.450698584 4681.93916666671
CONSENSUS_PEAKS_00000047 0 31 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000026 0 31 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_00000072 0 32 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000051 0 32 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_00000097 0 33 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000076 0 33 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_00000122 0 34 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000101 0 34 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_00000147 0 35 47809.7429166667 49486.7045833333 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000126 0 35 49486.7045833333 51163.66625 1676.96166666667
CONSENSUS_PEAKS_00000172 0 30 972887.450698584 974564.412365251 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000151 0 30 974564.412365251 976241.374031918 1676.96166666667
MACS2_ANNOTATE_00000034 0 31 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000059 0 32 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000084 0 33 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000109 0 34 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000134 0 35 51163.66625 52840.6279166667 1676.96166666667
MACS2_ANNOTATE_00000159 0 30 976241.374031918 977918.335698584 1676.96166666667
PHANTOMPEAKQUALTOOLS_00000027 0 31 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000052 0 32 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000077 0 33 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000102 0 34 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000127 0 35 52840.6279166667 60678.0225 7837.39458333333
PHANTOMPEAKQUALTOOLS_00000152 0 30 977918.335698584 985755.730281918 7837.39458333328
MAKE_GENOME_FILTER_00000020 0 6 419.240416666667 838.480833333333 419.240416666667
MERGED_BAM_FILTER_00000005 0 31 1542315.09820021 1547364.26236687 5049.16416666657
MACS2_00000006 0 31 1547364.26236687 1549041.22403354 1676.96166666667
BIGWIG_00000021 2 31 1549041.22403354 1553723.16320021 4681.93916666671
CONSENSUS_PEAKS_00000008 0 31 1553723.16320021 1555400.12486687 1676.96166666667
CONSENSUS_PEAKS_COUNTS_00000007 0 31 1555400.12486687 1557077.08653354 1676.96166666667
MACS2_ANNOTATE_00000023 0 31 1557077.08653354 1558754.04820021 1676.96166666667
PHANTOMPEAKQUALTOOLS_00000014 0 31 1558754.04820021 1566591.44278354 7837.39458333328
PLOTPROFILE_00000016 0 31 1566591.44278354 1575916.30736687 9324.86458333326
PLOTPROFILE_00000044 0 30 1043539.5595959 1052864.42417924 9324.86458333326
PLOTPROFILE_00000069 0 32 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000094 0 33 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000119 0 34 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000144 0 35 60678.0225 70002.8870833333 9324.86458333334
PLOTPROFILE_00000169 0 30 1052864.42417924 1062189.28876257 9324.86458333326
PRESEQ_00000013 0 30 1062189.28876257 1066051.9308459 3862.64208333334
PRESEQ_00000032 0 31 1575916.30736687 1579778.94945021 3862.64208333334
PRESEQ_00000057 0 32 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000082 0 33 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000107 0 34 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000132 0 35 70002.8870833333 73865.5291666667 3862.64208333334
PRESEQ_00000157 0 30 1066051.9308459 1069914.57292924 3862.64208333334
FASTQC_00000003 0 30 1069914.57292924 1072155.60167924 2241.02875000006
FASTQC_00000042 0 32 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000067 0 33 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000092 0 34 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000117 0 35 1912870.07901269 1915111.10776269 2241.02875000006
FASTQC_00000142 0 35 1915111.10776269 1917352.13651269 2241.02875000006
FASTQC_00000167 0 30 2027493.69707667 2029734.72582667 2241.02875000006
CONSENSUS_PEAKS_DESEQ2_00000029 0 18 1087572.13103642 1088690.10548086 1117.97444444452
CONSENSUS_PEAKS_DESEQ2_00000054 0 19 1043539.37286687 1044657.34731131 1117.97444444441
CONSENSUS_PEAKS_DESEQ2_00000079 0 20 1043539.37286687 1044657.34731131 1117.97444444441
CONSENSUS_PEAKS_DESEQ2_00000104 0 21 1043539.37286687 1044657.34731131 1117.97444444441
CONSENSUS_PEAKS_DESEQ2_00000129 0 22 139412.084251617 140530.058696061 1117.97444444444
CONSENSUS_PEAKS_DESEQ2_00000154 0 18 1643637.5920396 1644755.56648405 1117.97444444452
CONSENSUS_PEAKS_DESEQ2_00000018 0 31 1579778.94945021 1581455.91111687 1676.96166666667
PICARD_METRICS_00000015 0 31 1581455.91111687 1583132.87278354 1676.96166666667
PICARD_METRICS_00000033 0 31 1583132.87278354 1584809.83445021 1676.96166666667
PICARD_METRICS_00000058 0 32 1915111.10776269 1916788.06942935 1676.96166666667
PICARD_METRICS_00000083 0 33 1915111.10776269 1916788.06942935 1676.96166666667
PICARD_METRICS_00000108 0 34 1915111.10776269 1916788.06942935 1676.96166666667
PICARD_METRICS_00000133 0 35 1917352.13651269 1919029.09817935 1676.96166666667
PICARD_METRICS_00000158 0 30 2029734.72582667 2031411.68749334 1676.96166666667
PLOTFINGERPRINT_00000004 0 31 1599443.70076389 1601120.66243056 1676.96166666667
MACS2_QC_00000017 0 31 1601120.66243056 1602797.62409722 1676.96166666667
get_software_versions_00000019 0 7 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000035 0 19 1109588.53361392 1110706.50805836 1117.97444444452
PLOTFINGERPRINT_00000050 0 19 1175718.3302368 1176836.30468125 1117.97444444452
get_software_versions_00000045 0 8 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000060 0 22 1065555.77544437 1066673.74988882 1117.97444444452
PLOTFINGERPRINT_00000075 0 20 1131685.57206725 1132803.5465117 1117.97444444452
get_software_versions_00000070 0 9 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000085 0 23 1065555.77544437 1066673.74988882 1117.97444444452
PLOTFINGERPRINT_00000100 0 21 1131685.57206725 1132803.5465117 1117.97444444452
get_software_versions_00000095 0 10 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000110 0 24 1065555.77544437 1067232.73711104 1676.96166666667
PLOTFINGERPRINT_00000125 0 22 1131685.57206725 1132803.5465117 1117.97444444452
get_software_versions_00000120 0 11 419.240416666667 838.480833333333 419.240416666667
MACS2_QC_00000135 0 25 161428.48682912 163105.448495787 1676.96166666667
PLOTFINGERPRINT_00000150 0 25 227558.283452004 229235.24511867 1676.96166666667
get_software_versions_00000145 0 6 838.480833333333 1257.72125 419.240416666667
MACS2_QC_00000160 0 19 1665664.90280255 1666782.877247 1117.97444444452
PLOTFINGERPRINT_00000175 0 18 1731794.69942543 1732912.67386988 1117.97444444452
get_software_versions_00000170 0 7 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000010 0 31 1794363.77520209 1796040.73686875 1676.96166666667
IGV_00000022 0 20 1641955.67763947 1643073.65208392 1117.97444444452
CONSENSUS_PEAKS_ANNOTATE_00000024 0 19 1730200.06053686 1731318.03498131 1117.97444444452
output_documentation_00000025 0 8 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000041 0 8 1915117.79632754 1915537.03674421 419.240416666726
IGV_00000036 0 31 1796040.73686875 1797717.69853542 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000043 0 31 1797717.69853542 1799394.66020209 1676.96166666667
output_documentation_00000048 0 9 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000066 0 32 1916792.60642522 1918469.56809189 1676.96166666667
IGV_00000061 0 32 1918469.56809189 1920146.52975856 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000068 0 32 1920146.52975856 1921823.49142522 1676.96166666667
output_documentation_00000073 0 10 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000091 0 33 1916794.53592399 1918471.49759065 1676.96166666667
IGV_00000086 0 33 1918471.49759065 1920148.45925732 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000093 0 33 1920148.45925732 1921825.42092399 1676.96166666667
output_documentation_00000098 0 11 838.480833333333 1257.72125 419.240416666667
MULTIQC_00000116 0 34 1916796.46542275 1918473.42708942 1676.96166666667
IGV_00000111 0 34 1918473.42708942 1920150.38875608 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000118 0 34 1920150.38875608 1921827.35042275 1676.96166666667
output_documentation_00000123 0 26 0 1676.96166666667 1676.96166666667
MULTIQC_00000141 0 35 1919031.03034501 1920707.99201168 1676.96166666667
IGV_00000136 0 35 1920707.99201168 1922384.95367835 1676.96166666667
CONSENSUS_PEAKS_ANNOTATE_00000143 0 35 1922384.95367835 1924061.91534501 1676.96166666667
output_documentation_00000148 0 27 0 1676.96166666667 1676.96166666667
MULTIQC_00000166 0 30 2031413.619659 2033090.58132567 1676.96166666667
IGV_00000161 0 18 1883641.7532271 1884759.72767154 1117.97444444452
CONSENSUS_PEAKS_ANNOTATE_00000168 0 19 1927686.42998029 1928804.40442474 1117.97444444452
output_documentation_00000173 0 28 0 1676.96166666667 1676.96166666667
'''

from io import StringIO

s = StringIO(out_dyn)

dfdyn = pd.read_csv(s, sep=" ", names=["workflow", "variant", "processor", "from", "to", "duration"])
#print(df)


s = StringIO(out_stat)

dfstat = pd.read_csv(s, sep=" ", names=["workflow", "variant", "processor", "from", "to", "duration"])
#print(df)

merged_df = pd.merge(dfdyn, dfstat, on='workflow', how='outer')
#print(merged_df)

noteq = merged_df.loc[~(merged_df['processor_x'] == merged_df['processor_y'])]
noteq = noteq.loc[~(noteq['duration_x'] == noteq['duration_y'])]
print(noteq[['workflow','processor_x', 'processor_y', 'duration_x', 'duration_y', 'variant_x', 'variant_y']])

                              workflow  processor_x  processor_y  \
46          MERGED_BAM_FILTER_00000162           30           31   
47                      MACS2_00000028           31           18   
53                     BIGWIG_00000039           31           30   
59            CONSENSUS_PEAKS_00000047           31           19   
60     CONSENSUS_PEAKS_COUNTS_00000026           31           20   
71             MACS2_ANNOTATE_00000034           31           18   
84          MERGED_BAM_FILTER_00000005           31           30   
87            CONSENSUS_PEAKS_00000008           31           18   
88     CONSENSUS_PEAKS_COUNTS_00000007           31           19   
111                    FASTQC_00000167           30           12   
116    CONSENSUS_PEAKS_DESEQ2_00000129           22           24   
117    CONSENSUS_PEAKS_DESEQ2_00000154           18           31   
118    CONSENSUS_PEAKS_DESEQ2_00000018           31           19   
126           PLOTFINGERPRINT_00000004          

In [20]:
sorted_dfdyn = dfdyn.sort_values(['processor', 'from', 'to'])
pd.options.display.float_format = '{:.0f}'.format
print(sorted_dfdyn.to_string())


                              workflow  variant  processor    from      to  duration
21         MAKE_GENOME_FILTER_00000031        0          6       0     419       419
83         MAKE_GENOME_FILTER_00000020        0          6     419     838       419
143     get_software_versions_00000145        0          6     838    1258       419
22         MAKE_GENOME_FILTER_00000056        0          7       0     419       419
128     get_software_versions_00000019        0          7     419     838       419
146     get_software_versions_00000170        0          7     838    1258       419
23         MAKE_GENOME_FILTER_00000081        0          8       0     419       419
131     get_software_versions_00000045        0          8     419     838       419
150      output_documentation_00000025        0          8     838    1258       419
151                   MULTIQC_00000041        0          8 1915118 1915537       419
24         MAKE_GENOME_FILTER_00000106        0          9       

In [21]:
sorted_dfstat = dfstat.sort_values(['processor', 'from', 'to'])
pd.options.display.float_format = '{:.0f}'.format
print(sorted_dfstat.to_string())

                              workflow  variant  processor    from      to  duration
35         MAKE_GENOME_FILTER_00000020        0          6       0     419       419
41         MAKE_GENOME_FILTER_00000156        0          6     419     838       419
143     get_software_versions_00000145        0          6     838    1258       419
174      output_documentation_00000173        0          6    1258    1677       419
36         MAKE_GENOME_FILTER_00000031        0          7       0     419       419
128     get_software_versions_00000019        0          7     419     838       419
146     get_software_versions_00000170        0          7     838    1258       419
171                   MULTIQC_00000166        0          7 2310684 2311103       419
37         MAKE_GENOME_FILTER_00000056        0          8       0     419       419
131     get_software_versions_00000045        0          8     419     838       419
150      output_documentation_00000025        0          8     83